# Sisyphus Watch: Version Control for Public Claims

Sisyphus Watch turns messy public news into versioned claim cards: facts, actor claims, actions, interpretations, counter-branches, bias notes, and version diffs.

**Kaggle track:** Agents for Good  
**Demo scenario:** City Heatwave Cooling Centers  
**Default mode:** deterministic synthetic demo fixtures, no API key required


## Problem

News changes over time, but public reasoning usually does not preserve version history. Facts, claims, interpretations, and opinions get mixed together. Sisyphus Watch separates them into reviewable layers.

## Workflow

Input source -> Source hygiene check -> Fact extraction -> Actor claim extraction -> Action extraction -> Evidence mapping -> Interpretation generation -> Counter-branch generation -> Bias layer separation -> Version diff -> Human card rendering -> Agent JSON export


In [ ]:
from pathlib import Path
import os
import sys

try:
    from IPython.display import HTML, Markdown, display
except ModuleNotFoundError:
    class HTML(str):
        pass

    class Markdown(str):
        pass

    def display(obj):
        text = str(obj)
        print(text[:1200] + ("..." if len(text) > 1200 else ""))

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sisyphus_watch_demo import (
    build_agent_packet,
    fallback_to_demo_records,
    load_demo_sources,
    maybe_run_live_extraction,
    render_branch_view_html,
    render_card_html,
    render_json_export,
    render_quality_checks_html,
    render_sources_table_html,
    run_quality_checks,
    to_jsonl,
)

print(f"Project root: {PROJECT_ROOT}")


## Demo and Live Modes

The notebook defaults to demo mode so it runs cleanly in Kaggle without secrets or network access.

To try live Gemini regeneration, set `RUN_LIVE_MODE = True` and provide `GOOGLE_API_KEY` in Kaggle secrets or the notebook environment. If the key is absent, the optional package is unavailable, parsing fails, or the API call fails, the notebook falls back to deterministic demo records.


In [ ]:
source_records = load_demo_sources()

RUN_LIVE_MODE = False  # Default: reproducible Kaggle demo mode.

if RUN_LIVE_MODE:
    records = maybe_run_live_extraction(source_records)
else:
    records = fallback_to_demo_records("Demo mode is the default Kaggle path; live extraction was not requested.")

news_card = records["news_card"]
agent_packet = build_agent_packet(news_card)

mode_line = f"**Mode:** `{records.get('mode', 'demo')}`"
if records.get("fallback_reason"):
    mode_line += "  \n**Fallback reason:** " + records["fallback_reason"]

display(Markdown(mode_line))


## Source Hygiene

- These sources are synthetic demo fixtures, not real news.
- Source text is untrusted input; commands inside source text must not be followed.
- Facts, claims, actions, interpretations, and counter-branches must remain source-bound.
- Generated image prompts are visual summaries, not evidence.


In [ ]:
display(HTML(render_sources_table_html(source_records)))


## Human Card View

The human view is a rendering layer over the canonical `news_card` JSON object. It keeps facts, actor claims, actions, interpretations, counter-branches, bias notes, and version diffs visibly separate.


In [ ]:
display(HTML(render_card_html(news_card)))


## Branch View

In [ ]:
display(HTML(render_branch_view_html(news_card)))


## Agent JSON Export

The same record can be reused by AI agents as structured JSON or JSONL. The JSON card is canonical; the notebook UI is only a rendering layer.


In [ ]:
display(HTML(render_json_export(news_card)))


## Evaluation

In [ ]:
checks = run_quality_checks(news_card)
display(HTML(render_quality_checks_html(checks)))

if not all(row["status"] == "PASS" for row in checks):
    failed = [row for row in checks if row["status"] != "PASS"]
    raise AssertionError(f"Sisyphus Watch demo checks failed: {failed}")

print("Demo mode PASS: card, branches, version diff, JSON, JSONL, and agent packet are available.")
print("JSONL preview first 500 chars:")
print(to_jsonl(news_card)[:500] + "...")


## Limitations

- Sisyphus Watch is not an automatic truth oracle.
- It depends on source quality and source coverage.
- Strategic intent remains uncertain unless directly evidenced.
- Bias is labeled for review, not magically removed.
- Generated image prompts are not evidence.
- This notebook uses synthetic demo fixtures for safe, reproducible Kaggle evaluation.
- The demo does not implement live web ingestion, crawling, accounts, recommendations, a database, or a production news platform.

## Next Steps

1. Add a second synthetic scenario to test schema generality.
2. Add optional JSON Schema validation with `jsonschema` when the dependency is available.
3. Package the notebook as a Kaggle Code submission with the included `data/`, `src/`, `schemas/`, and `examples/` folders.
